In [28]:
from river import evaluate
from river import linear_model
from river.stream import iter_pandas
from river.metrics.base import Metrics
from river.metrics import MAE,MAPE,MSE,RMSE,base
from river import preprocessing
import pickle
import pandas as pd

In [2]:
data_path = '../results/FP_Truthful_Oracle_sigmoids_linucb_gaus_emb_prova/agent_stats_run_0_ctr_0.97_alpha_1.csv'
data = pd.read_csv(data_path)

In [3]:
y_df = data[['publisher', 'clicks', 'impressions', 'Iteration']]

In [4]:
embedding_path = '../src/publisher_embedding/data/embeddings_to_pick/pub_gaus_emb.pkl'
embeddings = pickle.load(open(embedding_path, 'rb'))

In [5]:
def dict_to_dataframe(dict):
    embedding_dim = list(dict.values())[0].shape[0]
    columns_list = ['publisher'] + [f'dim_{i}' for i in range(embedding_dim)]
    df = pd.DataFrame(columns=columns_list)
    for key, value in dict.items():
        df = pd.concat(
            [
                df,
                pd.DataFrame([
                    [key] + list(value)
                ], columns=columns_list)]
            ,
            ignore_index=True
        )
    return df

embeddings_df = dict_to_dataframe(embeddings)

/var/folders/pt/p9_0myf16lx1rjjrf63vkx5w0000gp/T/ipykernel_4901/592325403.py:6: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat(


In [6]:
training_data = pd.merge(
    y_df,
    embeddings_df,
    on='publisher',
    how='left'
)

In [8]:
embedding_dim = list(embeddings.values())[0].shape[0]
numerical_features = [f'dim_{i}' for i in range(embedding_dim)]

In [14]:
model = linear_model.LinearRegression(intercept_lr=.1)
stream = iter_pandas(X=training_data[numerical_features], y=training_data['clicks'])
metric = Metrics(metrics=[MAE()])
evaluate.progressive_val_score(
    dataset=stream,
    model=model,
    metric=metric,
    print_every=100
)

[100] MAE: 15.346496
[200] MAE: 16.228863
[300] MAE: 16.438372
[400] MAE: 16.720018
[500] MAE: 16.956673
[600] MAE: 16.801288
[700] MAE: 16.849458
[800] MAE: 16.904735
[900] MAE: 16.918164
[1,000] MAE: 16.984982
[1,100] MAE: 17.021926
[1,200] MAE: 17.081261
[1,300] MAE: 17.09968
[1,400] MAE: 17.116138
[1,500] MAE: 17.102374
[1,600] MAE: 17.100749
[1,700] MAE: 17.087466
[1,800] MAE: 17.054324
[1,900] MAE: 17.083552
[2,000] MAE: 17.065358
[2,100] MAE: 17.061869
[2,200] MAE: 17.061812
[2,300] MAE: 17.083793
[2,400] MAE: 17.076116
[2,500] MAE: 17.091966
[2,600] MAE: 17.114254
[2,700] MAE: 17.092222
[2,800] MAE: 17.096418
[2,900] MAE: 17.108086
[3,000] MAE: 17.109736
[3,100] MAE: 16.935464
[3,200] MAE: 16.706679
[3,300] MAE: 16.471186
[3,400] MAE: 16.220963
[3,500] MAE: 15.977849
[3,600] MAE: 15.76889
[3,700] MAE: 15.56346
[3,800] MAE: 15.374038
[3,900] MAE: 15.197815
[4,000] MAE: 15.02196
[4,100] MAE: 14.827951
[4,200] MAE: 14.629736
[4,300] MAE: 14.471449
[4,400] MAE: 14.263109
[4,500] MA

MAE: 13.15185

In [15]:
y_pred = model.predict_many(training_data[numerical_features])

In [32]:
lin_model = linear_model.LinearRegression()
# lin_model = (
#      preprocessing.StandardScaler() |
#      linear_model.LinearRegression()
# )
stream = iter_pandas(X=training_data[numerical_features], y=training_data['clicks'])

y_pred_list = []
for x, y in stream:
    y_pred = lin_model.predict_one(x)
    y_pred_list.append(y_pred)
    model = lin_model.learn_one(x, y)

In [33]:
all_data = data.copy()
all_data['clicks_pred'] = y_pred_list
all_data = all_data[['publisher', 'clicks', 'clicks_pred', 'est_clicks', 'Iteration']]

In [34]:
all_data[all_data['Iteration']>0].sort_values(by=['publisher', 'Iteration'])

,publisher,clicks,clicks_pred,est_clicks,Iteration
485,1001juegos.com,2.0,-7.482551,2.730745,1
785,1001juegos.com,2.0,-7.279743,2.904696,2
1085,1001juegos.com,0.0,-9.338066,3.426027,3
1385,1001juegos.com,0.0,-6.365852,3.445215,4
1685,1001juegos.com,2.0,-6.693713,3.646291,5
...,...,...,...,...,...
1768,zonawrestling.net,2.0,8.120166,8.023992,5
2068,zonawrestling.net,2.0,8.502050,8.085418,6
2368,zonawrestling.net,0.0,7.379979,8.112452,7
2668,zonawrestling.net,1.0,6.757476,8.134474,8
